# **CONNECT4 AGENT**

---



#
Imports

In [ ]:
# Imports needed for the entire notebook
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque
import random
import math
# For downloading parameters
import base64
from IPython.display import HTML, display
from google.colab import files # For loading parameters
import time

#
Game environment

In [ ]:
##############################
# Game logic and environment #
##############################

class Connect4:

    def __init__(self):
        self.rows = 6
        self.cols = 7
        self.reset()

    def reset(self):
        self.board = np.zeros((6, 7), dtype=int)
        self.current_player = 1 # Player 1 = 1, Player 2 = -1
        self.winner = None
        self.game_over = 0 # 0 = not over, 1 = win, -1 = draw
        self.last_move = None
        self.move_count = 0
        return self.get_state()

    def get_state(self):
        state = np.zeros((2, 6, 7), dtype=int)

        if self.current_player == 1:
            state[0] = (self.board == 1).astype(int) # Player 1 pieces
            state[1] = (self.board == -1).astype(int) # Player 2 pieces
        else:
            state[0] = (self.board == -1).astype(int) # Player 2 pieces
            state[1] = (self.board == 1).astype(int) # Player 1 pieces

        return state

    def get_valid_moves(self):
        return [col for col in range(7) if self.board[0, col] == 0]

    def get_lowest_row(self, col):
        for row in range(5, -1, -1):
            if self.board[row, col] == 0:
                return row
        return None

    def make_move(self, col):
        if self.game_over != 0:
            raise ValueError("Invalid move: Game is over!")

        if col not in self.get_valid_moves():
            raise ValueError(f"Invalid move: column {col} is full!")

        row = self.get_lowest_row(col)
        self.board[row, col] = self.current_player
        self.last_move = (row, col)
        self.move_count += 1
        reward = 0

        # check for win
        if self.check_winner(row, col, self.current_player):
            self.winner = self.current_player
            self.game_over = 1
            reward = 1
            return self.get_state(), reward, self.game_over

        # check for draw
        if self.move_count == 42:
            self.game_over = -1
            reward = 0
            return self.get_state(), reward, self.game_over

        self.current_player *= -1 # Switch player
        return self.get_state(), reward, self.game_over

    def check_winner(self, row, col, player):
        board = self.board

        # check horizontal
        for c in range(max(0, col-3), min(4, col+1)):
            if (board[row, c] == player and
                board[row, c+1] == player and
                board[row, c+2] == player and
                board[row, c+3] == player):
                return True

        # check vertical
        if row <= 2:
            if (board[row, col] == player and
                board[row+1, col] == player and
                board[row+2, col] == player and
                board[row+3, col] == player):
                return True

        # check diagonal \
        for offset in range(-3, 1):
            r = row + offset
            c = col + offset
            if 0 <= r and r <= 2 and 0 <= c and c <= 3:
                if (board[r, c] == player and
                    board[r+1, c+1] == player and
                    board[r+2, c+2] == player and
                    board[r+3, c+3] == player):
                    return True

        # check diagonal /
        for offset in range(-3, 1):
            r = row - offset
            c = col + offset
            if 3 <= r and r <= 5 and 0 <= c and c <= 3:
                if (board[r, c] == player and
                    board[r-1, c+1] == player and
                    board[r-2, c+2] == player and
                    board[r-3, c+3] == player):
                    return True

        return False

#
CNN

In [ ]:
####################
# CNN Architecture #
####################

class ResidualBlock(nn.Module):
    """Residual block with:
        - 2 conv layers + skip connection."""

    def __init__(self, channels): # channels: depth of data (ie. # of filters)
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels) # normalization for better learning
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        return F.relu(x)


class CNN(nn.Module):
    """
    AlphaZero-style architecture:
    - CNN with skip connections
    - Policy head: move winning probabilities (7 columns)
    - Value head: position winning advantage ([-1 = opponent advantage, 1 = current player advantage])
    We will refer to the notation in PUCT formula: https://substackcdn.com/image/fetch/$s_!8aNs!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fdbf3bcbe-836f-49b3-93d4-af0c48219f24_2212x1437.png
    """

    def __init__(self, num_res_blocks=5, channels=128):
        super().__init__()
        self.num_res_blocks = num_res_blocks
        self.channels = channels

        # Input layer (2D for each player's board pieces)
        self.conv_input = nn.Conv2d(2, channels, 3, padding=1, bias=False)
        self.bn_input = nn.BatchNorm2d(channels)

        # Residual blocks
        self.res_blocks = nn.ModuleList([ResidualBlock(channels) for _ in range(num_res_blocks)])

        # Policy head: P(s,a) move probabilities (s = state, a = action)
        self.policy_conv = nn.Conv2d(channels, 32, 1, bias=False)
        self.policy_bn = nn.BatchNorm2d(32)
        self.policy_fc = nn.Linear(32*6*7, 7)

        # Value head: V(s) position evaluation
        self.value_conv = nn.Conv2d(channels, 16, 1, bias=False)
        self.value_bn = nn.BatchNorm2d(16)
        self.value_fc1 = nn.Linear(16*6*7, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Input layer
        x = F.relu(self.bn_input(self.conv_input(x)))

        # Residual blocks
        for block in self.res_blocks:
            x = block(x)

        # Policy head: P(s,a) move probabilities
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        p = self.policy_fc(p)

        # Value head: V(s) position evaluation
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v)) # get activation range [-1, 1]

        return p, v

    def predict(self, state, valid_moves):
        """Predict policy P(s,a) and V(s) for current state."""
        self.eval()
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0)
            if next(self.parameters()).is_cuda:
                state_t = state_t.cuda()

            raw, value = self(state_t) # raw output: (1, 7), value: (1, 1)
            invalid_filter = torch.full((7,), float('-inf')) # Initialize valid moves with -inf

            # Set valid moves to 0
            invalid_filter[valid_moves] = 0

            if next(self.parameters()).is_cuda:
                invalid_filter = invalid_filter.cuda()

            # Valid moves: raw output + 0 = raw output
            # Invalid moves: raw output + (-inf) = -inf
            raw_1d = raw[0]
            valid_out = raw_1d + invalid_filter
            policy = F.softmax(valid_out, dim=0).cpu().numpy() # (-inf) -> 0

            return policy, value.item()

#
MCTS

In [ ]:
###############################################
# Monte Carlo Tree Search (MCTS) Architecture #
###############################################

class Node:
    """Notation:
    - N(s,a): # times node visited
    - W(s,a): total value accumulated
    - P(s,a): policy network previous probability
    """

    def __init__(self, prev):
        self.visit_count = 0 # N(s,a)
        self.val = 0         # W(s,a)
        self.prev = prev     # P(s,a)
        self.children = {}

    def value(self):
        if self.visit_count == 0:
            return 0
        return self.val / self.visit_count

    def has_children(self):
        return len(self.children) > 0


class MCTS:
    """Monte Carlo Tree Search. NN-guided selection and expansion phase. Uses PUCT formula.
    Additional notation:
    - Q(s,a): mean action value = W(s,a) / N(s,a)
    - c_puct: exploration constant
    """

    def __init__(self, network, c_puct=1.5, num_simulations=200):
        self.network = network
        self.c_puct = c_puct
        self.num_simulations = num_simulations

    def search(self, game, temp=1.0):
        root = Node(prev=0)
        # Initialize root with NN policy probabilities P(s,a)
        state = game.get_state()
        valid_moves = game.get_valid_moves()
        policy, _ = self.network.predict(state, valid_moves)

        for action in valid_moves:
            root.children[action] = Node(prev=policy[action])

        # MCTS simulations
        for _ in range(self.num_simulations):
            node = root
            sim_game = self.copy_game(game)
            total_path = [node]

            # Selection using PUCT
            while node.has_children() and not sim_game.game_over:
                action, node = self.PUCT_select(node)
                sim_game.make_move(action)
                total_path.append(node)

            # Expansion + Evaluation
            if sim_game.game_over:
                if sim_game.game_over == 1:
                    value = -1 # current player loss
                else:
                    value = 0  # draw
            else:
            # NN evaluation when game not over
                state = sim_game.get_state()
                valid_moves = sim_game.get_valid_moves()
                policy, value = self.network.predict(state, valid_moves)

                # Add children with prior probabilities P(s,a) from policy network
                for action in valid_moves:
                    node.children[action] = Node(prev=policy[action])

            self.update_path(total_path, value) # back propagate the path and update visited node attributes

        return self.get_probs(root, valid_moves, temp) # return new move probabilities based on visit counts

    def copy_game(self, game):
        new_game = Connect4()
        new_game.board = game.board.copy()
        new_game.current_player = game.current_player
        new_game.game_over = game.game_over
        new_game.winner = game.winner
        new_game.move_count = game.move_count
        return new_game

    def PUCT_select(self, node):
        """Return action and child with best PUCT score"""
        best_score = float('-inf')
        best_action = None
        best_child = None

        for action, child in node.children.items():
            score = self.PUCT(node, child)
            if score > best_score:
                best_score = score
                best_action = action
                best_child = child

        return best_action, best_child

    def PUCT(self, parent, child): # PUCT formula
        prev_score = (self.c_puct * child.prev * math.sqrt(parent.visit_count) / (1 + child.visit_count))
        value_score = -child.value() if child.visit_count > 0 else 0 # value_score = Q(s,a)
        return value_score + prev_score

    def update_path(self, search_path, value):
        """Backpropagate W(s,a) and N(s,a) through search path."""
        for node in reversed(search_path):
            node.val += value
            node.visit_count += 1
            value = -value

    def get_probs(self, root, valid_moves, temp):
        """Compute updated move probabilities from visit counts. Formula from the AlphaGo Zero paper: π(a) ∝ N(s,a)^(1/t) with temp = t"""
        visits = np.array([root.children[a].visit_count if a in root.children else 0 for a in range(7)])

        if temp == 0:
            probs = np.zeros(7)
            probs[np.argmax(visits)] = 1
        else:
            visits_temp = visits ** (1 / temp)
            probs = visits_temp / visits_temp.sum()

        return probs

#
Buffer

In [ ]:
#################
# Replay Buffer #
#################

class ReplayBuffer:
    """Buffer of capacity 100000.Training sample shape: (state, π, game outcome) where outcome = (1:win, -1:loss, 0:draw)"""

    def __init__(self, max_size=100000):
        self.buffer = deque(maxlen=max_size)

    def push(self, state, policy, value, flip=True): # Flip = add horizontal flip for each state
        self.buffer.append((state, policy, value))

        if flip == True:
            flipped_state = np.flip(state, axis=2).copy()   # Flip columns
            flipped_policy = np.flip(policy).copy()         # Flip policy
            self.buffer.append((flipped_state, flipped_policy, value))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, min(batch_size, len(self.buffer)))
        states, policies, values = zip(*batch)
        return (np.array(states), np.array(policies), np.array(values))

# TRAINING MODEL

---



In [ ]:
############
# Training #
############

class Trainer:

    def __init__(self, network, lr=0.001, wd=0.0001, number_simulations=200):
        self.network = network
        self.mcts = MCTS(network, num_simulations=number_simulations)
        self.buffer = ReplayBuffer()
        self.optimizer = optim.Adam(network.parameters(), lr=lr, weight_decay=wd)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.network.to(self.device)

    def self_play(self):
        """Self-play one game and use MCTS to improve policy at each step."""
        game = Connect4()
        history = []

        while not game.game_over:
            state = game.get_state()
            temp = 1.0 if game.move_count < 15 else 0.1 # threshold at 15 moves for exploration vs exploitation
            policy = self.mcts.search(game, temp=temp)
            history.append((state.copy(), policy, game.current_player))

            action = np.random.choice(7, p=policy) # sample action from policy and play it
            game.make_move(action)

        # Assign values to each state in the history
        training_data = []
        for state, policy, player in history:
            if game.winner is None:
                value = 0 # draw
            elif game.winner == player:
                value = 1 # win
            else:
                value = -1 # loss
            training_data.append((state, policy, value))

        return training_data

    def train_step(self, batch_size=64):
        """Train network to predict π and game outcome on a batch of samples from replay buffer.
        Policy loss: cross-entropy between predicted π and MCTS improved π.
        Value loss: MSE between predicted V(s) and game outcome.
        """
        if len(self.buffer) < batch_size:
            return None

        self.network.train()
        states, policies, values = self.buffer.sample(batch_size)

        states_t = torch.FloatTensor(states).to(self.device)
        policies_t = torch.FloatTensor(policies).to(self.device)
        values_t = torch.FloatTensor(values).unsqueeze(1).to(self.device)

        pred_output, pred_values = self.network(states_t)
        # Policy loss
        policy_loss = -torch.sum(policies_t * F.log_softmax(pred_output, dim=1)) / batch_size
        # Value Loss
        value_loss = F.mse_loss(pred_values, values_t)
        # Total loss to backpropagate on
        loss = policy_loss + value_loss

        # Backpropagation + optimization
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {"total_loss": loss.item(), "policy_loss": policy_loss.item(), "value_loss": value_loss.item()}

    def train(self, num_iterations=150, games_per_iter=30, train_steps_per_iter=200, batch_size=128):
        print("Training...")
        print("Specs:")
        print(f"Network: {self.network.num_res_blocks} res blocks, {self.network.channels} channels")
        print(f"Num MCTS simulations: {self.mcts.num_simulations}")
        print(f"Iterations: {num_iterations}, num games: {games_per_iter}")
        print("\n")

        for iteration in range(num_iterations):
            print(f"\nIteration {iteration + 1}/{num_iterations}")
            print("Self-play:")
            draws = 0

            for g in range(games_per_iter):
                game_hist = self.self_play()
                for state, policy, value in game_hist:
                    self.buffer.push(state, policy, value, flip=True)

                if game_hist[-1][2] == 1: # Player already won
                    pass
                elif game_hist[-1][2] == 0:
                    draws += 1

                if (g + 1) % 10 == 0:
                    print(f"Completed {g + 1}/{games_per_iter} games")
            print(f"Buffer size: {len(self.buffer)}")

            # Training
            print(f"Training: {train_steps_per_iter} steps.")
            total_loss = 0
            total_policy_loss = 0
            total_value_loss = 0

            for _ in range(train_steps_per_iter):
                losses = self.train_step(batch_size)
                if losses:
                    total_loss += losses["total_loss"]
                    total_policy_loss += losses["policy_loss"]
                    total_value_loss += losses["value_loss"]

            avg_loss = total_loss / train_steps_per_iter
            avg_policy = total_policy_loss / train_steps_per_iter
            avg_value = total_value_loss / train_steps_per_iter
            print(f"Avg Loss: {avg_loss:.4f}, Avg Policy: {avg_policy:.4f}, Avg Value: {avg_value:.4f}")

            # Validation
            if (iteration + 1) % 10 == 0:
                self.evaluate()

            # Saves model parameters every 10 iterations (this works with the VSCode Colab extension too)
            if (iteration + 1) % 10 == 0:
                checkpoint_path = f"connect4_checkpoint_{iteration+1}.pth"
                torch.save(self.network.state_dict(), checkpoint_path)
                # Download link
                try:
                    with open(checkpoint_path, "rb") as f:
                        model_bytes = f.read()
                    b64 = base64.b64encode(model_bytes).decode()
                    href = f'<a download="{checkpoint_path}" href="data:application/octet-stream;base64,{b64}">Download {checkpoint_path}</a>'
                    display(HTML(href))
                except:
                    pass

    def evaluate(self, num_games=20):
        """Plays against random moves and prints win rate"""
        wins, losses, draws = 0, 0, 0
        for _ in range(num_games):
            game = Connect4()
            ai_player = random.choice([1, -1])

            while not game.game_over:
                if game.current_player == ai_player:
                    policy = self.mcts.search(game, temp=0)
                    action = np.argmax(policy)
                else:
                    action = random.choice(game.get_valid_moves())
                game.make_move(action)

            if game.winner == ai_player:
                wins += 1
            elif game.winner == -ai_player:
                losses += 1
            else:
                draws += 1
        print(f"Evaluation vs random: {wins}W / {losses}L / {draws}D - {100*wins/num_games:.1f}% win rate")

In [ ]:
#############################################
# Smaller Model Architecture (1rst attempt) #
#############################################

network = CNN(num_res_blocks=4, channels=64) # ~200k parameters
trainer = Trainer(network, lr=0.001, number_simulations=100)
trainer.train(num_iterations=50, games_per_iter=25, train_steps_per_iter=100, batch_size=64)

Training...
Specs:
Network: 4 res blocks, 64 channels
Num MCTS simulations: 100
Iterations: 50, num games: 25



Iteration 1/50
Self-play:
Completed 10/25 games
Completed 20/25 games


TypeError: object of type 'ReplayBuffer' has no len()

In [ ]:
############################
# Large Model Architecture #
############################

network = CNN(num_res_blocks=5, channels=128) # ~1.6M parameters
trainer = Trainer(network, lr=0.001, number_simulations=200)
trainer.train(num_iterations=70, games_per_iter=30, train_steps_per_iter=200, batch_size=128)

# LOADING MODEL PARAMETERS

---



In [ ]:
##############################
# Smaller Model Architecture #
##############################

uploaded = files.upload() # pick the corresponding parameters file from your computer

trained_network = CNN(num_res_blocks=4, channels=64)
trained_network.load_state_dict(torch.load(uploaded.filename, map_location='cpu'))
trained_network.eval()

In [ ]:
############################
# Large Model Architecture #
############################

uploaded = files.upload() # pick the corresponding parameters file from your computer

trained_network = CNN(num_res_blocks=5, channels=128)
trained_network.load_state_dict(torch.load("connect4_checkpoint_50.pth", map_location='cpu'))
trained_network.eval()


Saving connect4_checkpoint_50.pth to connect4_checkpoint_50.pth


CNN(
  (conv_input): Conv2d(2, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn_input): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (res_blocks): ModuleList(
    (0-4): 5 x ResidualBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (policy_conv): Conv2d(128, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (policy_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (policy_fc): Linear(in_features=1344, out_features=7, bias=True)
  (value_conv): Conv2d(128, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (value_bn): BatchNorm2d(16, eps=1e-05

In [ ]:

mc = MCTS(trained_network, num_simulations=400)
game = Connect4()
game.make_move(3,)

move = np.argmax(mc.search(game, temp=0))
print(move)
game.make_move(move)

game.make_move(5)

move = np.argmax(mc.search(game, temp=0))
print(move)

game.make_move(move)

game.make_move(5)

move = np.argmax(mc.search(game, temp=0))
print(move)

game.make_move(move)

3
4
3


(array([[[0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0],
         [0, 0, 0, 1, 0, 1, 0]],
 
        [[0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 0, 1, 0, 0]]]),
 0,
 0)